In [2]:
import pandas as pd
import joblib 

X_test = pd.read_csv("../data/selected/X_test_selected.csv")
y_test = pd.read_csv("../data/selected/y_test.csv").squeeze()
print("Test shape:", X_test.shape)
print("Target shape:", y_test.shape)

print("Features:")
print(X_test.columns.tolist())

print("\nTarget distribution:")
print(y_test.value_counts())

Test shape: (2997, 15)
Target shape: (2997,)
Features:
['n_slash', 'n_hypens', 'n_redirection', 'count', 'n_equal', 'url_length', 'n_and', 'n_dots', 'n_questionmark', 'n_at', 'n_plus', 'n_space', 'n_percent', 'n_dollar', 'n_asterisk']

Target distribution:
phishing
1    2381
0     616
Name: count, dtype: int64


In [3]:
lr_model = joblib.load("../models/logistic_regression.pkl")
dt_model = joblib.load("../models/decision_tree.pkl")
rf_model = joblib.load("../models/random_forest.pkl")
print("All three models loaded successfully.")

#sanity check to be removed
print("Logistic Regression:", type(lr_model))
print("Decision Tree:", type(dt_model))
print("Random Forest:", type(rf_model))

All three models loaded successfully.
Logistic Regression: <class 'sklearn.linear_model._logistic.LogisticRegression'>
Decision Tree: <class 'sklearn.tree._classes.DecisionTreeClassifier'>
Random Forest: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [4]:
print("Test features:", X_test.shape[1])
assert X_test.shape[1] == 15
print("Feature count verified.")

Test features: 15
Feature count verified.


In [11]:
lr_pred = lr_model.predict(X_test)
dt_pred = dt_model.predict(X_test)
rf_pred = rf_model.predict(X_test)
print("Predictions generated successfully.")

print("LR predictions:", lr_pred.shape)
print("DT predictions:", dt_pred.shape)
print("RF predictions:", rf_pred.shape)

lr_proba = lr_model.predict_proba(X_test)
dt_proba = dt_model.predict_proba(X_test)
rf_proba = rf_model.predict_proba(X_test)
print("\nProbabilities generated successfully.")

print("LR probability shape:", lr_proba.shape)
print("DT probability shape:", dt_proba.shape)
print("RF probability shape:", rf_proba.shape)

lr_prob = lr_proba[:, 1]
dt_prob = dt_proba[:, 1]
rf_prob = rf_proba[:, 1]
print("\nLR probability range:", lr_prob.min(), "→", lr_prob.max())
print("DT probability range:", dt_prob.min(), "→", dt_prob.max())
print("RF probability range:", rf_prob.min(), "→", rf_prob.max())

results = pd.DataFrame({
    "actual": y_test.values,
    "lr_prediction": lr_pred,
    "dt_prediction": dt_pred,
    "rf_prediction": rf_pred,
    "lr_probability": lr_prob,
    "dt_probability": dt_prob,
    "rf_probability": rf_prob
})

print("\n",results.head())
#sanity check
print("\nLR classes:", lr_model.classes_)
print("DT classes:", dt_model.classes_)
print("RF classes:", rf_model.classes_)



Predictions generated successfully.
LR predictions: (2997,)
DT predictions: (2997,)
RF predictions: (2997,)

Probabilities generated successfully.
LR probability shape: (2997, 2)
DT probability shape: (2997, 2)
RF probability shape: (2997, 2)

LR probability range: 5.373757232776395e-05 → 1.0
DT probability range: 0.0 → 1.0
RF probability range: 0.0 → 1.0

    actual  lr_prediction  dt_prediction  rf_prediction  lr_probability  \
0       1              1              1              1        0.995459   
1       0              0              0              0        0.482308   
2       1              1              1              1        0.996116   
3       1              1              1              1        0.998464   
4       1              1              1              1        0.994614   

   dt_probability  rf_probability  
0             1.0            1.00  
1             0.0            0.08  
2             1.0            0.99  
3             1.0            1.00  
4             1

In [12]:
#model agreement and disagreement counts 
results["all_agree"] = (
    (results["lr_prediction"] == results["dt_prediction"]) &
    (results["dt_prediction"] == results["rf_prediction"])
)

print("All three agree:", results["all_agree"].sum())
print("Total samples:", len(results))
print("Agreement rate:", results["all_agree"].mean())
print("Disagreements:", (~results["all_agree"]).sum())

All three agree: 2456
Total samples: 2997
Agreement rate: 0.8194861528194861
Disagreements: 541


In [13]:
results["prediction_pattern"] = (
    results["lr_prediction"].astype(str)
    + "-"
    + results["dt_prediction"].astype(str)
    + "-"
    + results["rf_prediction"].astype(str)
)

print(results["prediction_pattern"].value_counts())
disagreements = results[~results["all_agree"]].copy()

print("\nNumber of disagreement cases:", len(disagreements))

disagreements.head(10)

prediction_pattern
1-1-1    2184
0-0-0     272
1-0-0     209
1-0-1     140
0-1-1      96
1-1-0      47
0-1-0      35
0-0-1      14
Name: count, dtype: int64

Number of disagreement cases: 541


,actual,lr_prediction,dt_prediction,rf_prediction,lr_probability,dt_probability,rf_probability,all_agree,prediction_pattern
9,1,1,0,0,0.718929,0.0,0.042333,False,1-0-0
18,0,1,0,0,0.508203,0.0,0.110000,False,1-0-0
27,1,1,1,0,0.835362,1.0,0.420000,False,1-1-0
31,0,1,0,0,0.637980,0.0,0.170000,False,1-0-0
33,1,1,0,0,0.530893,0.0,0.235667,False,1-0-0
35,1,1,1,0,0.744030,1.0,0.420000,False,1-1-0
36,1,0,1,1,0.370381,1.0,0.890000,False,0-1-1
47,1,0,1,1,0.392966,1.0,0.720000,False,0-1-1
50,1,0,0,1,0.325116,0.0,0.670000,False,0-0-1
54,1,0,1,1,0.379686,1.0,0.710000,False,0-1-1


In [14]:
disagreements[
    [
        "actual",
        "lr_prediction",
        "dt_prediction",
        "rf_prediction",
        "lr_probability",
        "dt_probability",
        "rf_probability"
    ]
].head(20)

,actual,lr_prediction,dt_prediction,rf_prediction,lr_probability,dt_probability,rf_probability
9,1,1,0,0,0.718929,0.0,0.042333
18,0,1,0,0,0.508203,0.0,0.110000
27,1,1,1,0,0.835362,1.0,0.420000
31,0,1,0,0,0.637980,0.0,0.170000
33,1,1,0,0,0.530893,0.0,0.235667
35,1,1,1,0,0.744030,1.0,0.420000
36,1,0,1,1,0.370381,1.0,0.890000
47,1,0,1,1,0.392966,1.0,0.720000
50,1,0,0,1,0.325116,0.0,0.670000
54,1,0,1,1,0.379686,1.0,0.710000


In [15]:
print("Prediction patterns:")
print(results["prediction_pattern"].value_counts())
print("\nDisagreement pattern counts:")
print(
    results.loc[~results["all_agree"], "prediction_pattern"]
    .value_counts()
)
disagreements["correct_models"] = (
    (disagreements["lr_prediction"] == disagreements["actual"]).astype(int)
    + (disagreements["dt_prediction"] == disagreements["actual"]).astype(int)
    + (disagreements["rf_prediction"] == disagreements["actual"]).astype(int)
)

print(disagreements["correct_models"].value_counts().sort_index())

Prediction patterns:
prediction_pattern
1-1-1    2184
0-0-0     272
1-0-0     209
1-0-1     140
0-1-1      96
1-1-0      47
0-1-0      35
0-0-1      14
Name: count, dtype: int64

Disagreement pattern counts:
prediction_pattern
1-0-0    209
1-0-1    140
0-1-1     96
1-1-0     47
0-1-0     35
0-0-1     14
Name: count, dtype: int64
correct_models
1    161
2    380
Name: count, dtype: int64
